# RSNA: COATNET · Steven 4,000 · five models

Run code cells **1–6** for setup and gold caching, **7** to train all five models, **8** for the ensemble submission. Attach the competition and [Steven's public labels](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels). No MedGemma CSV or API is needed. Install missing packages in Kaggle before cell 1: `pip install timm safetensors pydicom opencv-python-headless scikit-learn` (compressed DICOM may additionally need `pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg`).

- Deterministic 4,000 non-gold Steven v2 studies, five **disjoint groups of 800**. Each independent model sees only its own group.
- Gold 58 split into five validation folds (12/12/12/11/11); remaining gold studies train with three repetitions per epoch. Validation gold never enters that model's loss.
- Cache all 800 assigned studies to disk **before** the model trains, reuse for five full epochs, then delete that group's pixels. Gold disk cache stays and its pixels are also loaded into RAM once. Failed/interrupted training cleans the active group.
- Centre crop a **140 mm square** using row/column DICOM PixelSpacing, pad when needed, resize to 224. Missing/implausible spacing uses the central 80% square. Physical ordering, slope/intercept, MONOCHROME1 inversion, series percentile 1–99 normalization remain. Physical cropping reference: [public competition model](https://www.kaggle.com/dreaddevelopment/raptor-knee-widedense).
- **10 positions per selected series**, spread across 5–95%; each input has three adjacent slices as channels. Up to six selected series (two per plane), so at most 60 encoder inputs per study. This means 10 triplets, not 10 individual DICOM files. Fixed positions allow small disk caches; training retains mild consistent rotation/translation/intensity augmentation. Validation/test have none.
- Standard ImageNet initialization, final backbone stage only fine-tuned, backbone/head LR `3e-5`/`1e-4`, cosine decay, `.75` weak loss with `2*abs(p-.5)` confidence weights. Best gold BCE checkpoint per model; average five model predictions for submission.

At six series per study the active 800-study pixel payload is at most **6.73 GiB** plus gold and checkpoints (actual series count may be lower). A 2 GiB disk reserve stops construction safely. Training reads cached `.npy` pixels; it does not decode DICOM each epoch. Ten triplets are fixed across epochs, with fresh augmentation. Avoid running the two notebooks concurrently on the same GPU.

Both notebooks use identical selected IDs and folds for comparison. This is intentionally partitioned weak training, not conventional five-fold training on all remaining weak studies. Steven's upstream labels used gold calibration/evaluation, so image-model OOF is not completely independent of label development. Full Kaggle GPU training has not been run here.


In [1]:
from pathlib import Path
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor
import time, random, gc
import numpy as np
import pandas as pd
import torch, cv2, pydicom, timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark=True
LABEL_COLS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
IMAGE_SIZE=224; CENTERS_PER_PLANE=10; IMAGE_MICROBATCH=5
NUM_DICOM_THREADS=4; NUM_WORKERS=2; N_FOLDS=5; EPOCHS=5
PSEUDO_LOSS_WEIGHT=.75; EXPECTED_PSEUDO_COUNT=4000
WEAK_LABEL_SOURCE='steven'
STEVEN_LABEL_CSV=Path('/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v2.csv')
MODEL_NAME='coatnet_0_rw_224.sw_in1k'
assert WEAK_LABEL_SOURCE in ('steven','medgemma')
RUN_TAG=f"coatnet_base_{WEAK_LABEL_SOURCE}{EXPECTED_PSEUDO_COUNT}_w{int(round(PSEUDO_LOSS_WEIGHT*100)):02d}"
LR_BACKBONE=3e-5; LR_HEAD=1e-4
WEAK_PASSES=EPOCHS; GOLD_REPEATS=3; MIN_LR_RATIO=.1
SLICE_JITTER_FRACTION=0.  # Fixed slice triplets are disk-cached; augmentation remains training-only.
CENTER_CROP_MM=140.; FALLBACK_CENTER_FRACTION=.8
AUG_ROTATION_DEG=5.; AUG_TRANSLATION=.03
AUG_GAIN=(.9,1.1); AUG_OFFSET=.03
RUN_TAG=RUN_TAG+'_weakpart5_5epoch_crop140mm_10slices'
DATA_CANDIDATES=[Path('/kaggle/input/rsna-knee-abnormality-detection'),Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
DATA_ROOT=next((p for p in DATA_CANDIDATES if p.exists()),None)
if DATA_ROOT is None: raise FileNotFoundError('Attach RSNA Knee Abnormality Detection competition data.')
TRAIN_SERIES_ROOT=DATA_ROOT/'train_series'; TEST_SERIES_ROOT=DATA_ROOT/'test_series'
OUT=Path('/kaggle/working')
CACHE_ROOT=OUT/'rsna_coatnet_crop140mm_10triplet_cache_v2'
CACHE_MIN_FREE_GIB=2.0  # Stop cache build before consuming checkpoint space.
TEST_CACHE_GROUP_SIZE=400  # Test-only storage group limit.
CACHE_BUILD_WORKERS=2
CLEAR_OLD_WEAK_CACHE=True  # Clean abandoned pixel files only inside this notebook cache directory.
COATNET_WEIGHTS= Path("/kaggle/input/models/amritgopinath/coatnet-model/pytorch/default/1/model.safetensors")  # Optional local STANDARD ImageNet checkpoint; never the knee-pretrained file.
print('device',DEVICE,'timm',timm.__version__)


device cuda timm 1.0.29


In [2]:
# Download only the standard timm ImageNet model once, or use an explicit local file.
BASE_CACHE=OUT/'coatnet_0_rw_224_sw_in1k_base.pt'
if COATNET_WEIGHTS is not None:
    COATNET_WEIGHTS=Path(COATNET_WEIGHTS)
    if not COATNET_WEIGHTS.is_file(): raise FileNotFoundError(COATNET_WEIGHTS)
    if 'coatnet_kneemri' in COATNET_WEIGHTS.name.lower() or 'coatnet-fine-tuned' in str(COATNET_WEIGHTS):
        raise ValueError('This experiment requires standard ImageNet weights, not the knee-pretrained model.')
elif BASE_CACHE.is_file():
    COATNET_WEIGHTS=BASE_CACHE
else:
    try:
        base=timm.create_model(MODEL_NAME,pretrained=True)
    except Exception as exc:
        raise RuntimeError('Enable Internet for the initial standard ImageNet download, or attach timm/coatnet_0_rw_224.sw_in1k weights and set COATNET_WEIGHTS in cell 1.') from exc
    torch.save(base.state_dict(),BASE_CACHE)
    del base
    COATNET_WEIGHTS=BASE_CACHE
print('Base ImageNet checkpoint:',COATNET_WEIGHTS)


Base ImageNet checkpoint: /kaggle/input/models/amritgopinath/coatnet-model/pytorch/default/1/model.safetensors


In [3]:
train=pd.read_csv(DATA_ROOT/'train.csv',dtype={'StudyInstanceUID':str})
train_series=pd.read_csv(DATA_ROOT/'train_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
test_series=pd.read_csv(DATA_ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
sample_submission=pd.read_csv(DATA_ROOT/'sample_submission.csv',dtype={'StudyInstanceUID':str})
assert list(sample_submission.columns)==['StudyInstanceUID']+LABEL_COLS
gold=train[train[LABEL_COLS].notna().all(axis=1)].copy().reset_index(drop=True)
gold[LABEL_COLS]=gold[LABEL_COLS].apply(pd.to_numeric)
assert len(gold)==58 and gold.StudyInstanceUID.is_unique
if not STEVEN_LABEL_CSV.is_file():
    candidates=sorted(set(p for pattern in ('*/llm_labels_v2.csv','datasets/*/*/llm_labels_v2.csv')
                          for p in Path('/kaggle/input').glob(pattern)))
    if len(candidates)!=1: raise FileNotFoundError('Attach Steven labels and set STEVEN_LABEL_CSV in cell 1.')
    STEVEN_LABEL_CSV=candidates[0]
labels=pd.read_csv(STEVEN_LABEL_CSV,dtype={'StudyInstanceUID':str})
assert {'StudyInstanceUID',*LABEL_COLS}.issubset(labels.columns)
assert labels.StudyInstanceUID.notna().all() and labels.StudyInstanceUID.is_unique
eligible=set(train.StudyInstanceUID)&set(train_series.StudyInstanceUID)-set(gold.StudyInstanceUID)
labels=labels[labels.StudyInstanceUID.isin(eligible)].sort_values('StudyInstanceUID').reset_index(drop=True)
for col in LABEL_COLS:
    labels[col]=pd.to_numeric(labels[col],errors='raise')
    assert np.isfinite(labels[col]).all() and labels[col].between(0,1).all(),col
assert len(labels)>=EXPECTED_PSEUDO_COUNT, f'Need 4000 non-gold studies; found {len(labels)}'
pseudo=labels.sample(n=EXPECTED_PSEUDO_COUNT,random_state=SEED).reset_index(drop=True)
for col in LABEL_COLS: pseudo['weight__'+col]=2*(pseudo[col]-.5).abs()
assert len(pseudo)==4000 and pseudo.StudyInstanceUID.is_unique
assert set(pseudo.StudyInstanceUID).isdisjoint(gold.StudyInstanceUID)
import hashlib,json
selected_hash=hashlib.sha256('\n'.join(sorted(pseudo.StudyInstanceUID)).encode()).hexdigest()
pseudo.to_csv(OUT/f'{RUN_TAG}_training_labels_4000.csv',index=False)
manifest={'model':MODEL_NAME,'initialization':'standard ImageNet; no knee pretraining',
          'label_file':str(STEVEN_LABEL_CSV),'label_file_sha256':hashlib.sha256(STEVEN_LABEL_CSV.read_bytes()).hexdigest(),
          'selected_ids_sha256':selected_hash,'weak_studies':len(pseudo),'gold_studies':len(gold),
          'folds':N_FOLDS,'weak_per_model':800,'epochs':EPOCHS,'seed':SEED,'crop_mm':CENTER_CROP_MM,
          'centers_per_series':CENTERS_PER_PLANE,'weak_loss_weight':PSEUDO_LOSS_WEIGHT}
(OUT/f'{RUN_TAG}_experiment.json').write_text(json.dumps(manifest,indent=2))
print('Gold:',len(gold),'eligible Steven:',len(labels),'selected:',len(pseudo),'selection hash:',selected_hash)


Gold: 58 eligible Steven: 4349 selected: 4000 selection hash: 323bf914042ea4f22104953c97288dcc299b995ea08731240225f5db65f0a9fc


In [4]:
from safetensors.torch import load_file as load_safetensors

def make_backbone(load_pretrained=True):
    net=timm.create_model(MODEL_NAME,pretrained=False)
    expected_size=int(timm.data.resolve_model_data_config(net)['input_size'][-1])
    if IMAGE_SIZE != expected_size:
        raise ValueError(f'IMAGE_SIZE={IMAGE_SIZE} is incompatible with this CoAtNet checkpoint/model; expected {expected_size}.')
    net.reset_classifier(0)  # Match a feature-only or full ImageNet checkpoint.
    if load_pretrained and COATNET_WEIGHTS.suffix=='.safetensors': state=load_safetensors(str(COATNET_WEIGHTS),device='cpu')
    elif load_pretrained:
        state=torch.load(COATNET_WEIGHTS,map_location='cpu',weights_only=True)
        if isinstance(state,dict): state=state.get('state_dict',state.get('model',state))
        state={k.removeprefix('module.'):v for k,v in state.items()}
    if load_pretrained:
        # Classifier keys are optional and unused for the 12-label task.
        # Keep strict=True for every remaining backbone weight and shape.
        state={k.removeprefix('module.'):v for k,v in state.items()}
        for key in ('head.fc.weight','head.fc.bias'):
            state.pop(key,None)
        net.load_state_dict(state,strict=True)
    for p in net.parameters(): p.requires_grad_(False)
    if not hasattr(net,'stages'): raise RuntimeError('Unexpected timm CoAtNet layout; inspect net and choose its final stage.')
    for p in net.stages[-1].parameters(): p.requires_grad_(True)
    return net

class StudyCoAtNet(nn.Module):
    def __init__(self,load_pretrained=True):
        super().__init__()
        self.backbone=make_backbone(load_pretrained)
        dim=self.backbone.num_features
        self.head=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(.15),nn.Linear(dim,len(LABEL_COLS)))
        self.plane_weight=nn.Parameter(torch.zeros(4,len(LABEL_COLS)))
        self.register_buffer('mean',torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std',torch.tensor([.229,.224,.225]).view(1,3,1,1))

    def train_backbone_tail(self):
        self.backbone.eval()
        self.backbone.stages[-1].train()

    def forward(self,images,planes):
        slice_logits=[]
        for chunk in images.split(IMAGE_MICROBATCH):
            x=(chunk.to(dtype=self.mean.dtype)/255-self.mean)/self.std
            ctx=torch.autocast('cuda',dtype=torch.float16) if self.training and images.is_cuda else nullcontext()
            with ctx: slice_logits.append(self.head(self.backbone(x)))
        scores=torch.cat(slice_logits,0)
        pooled=[]; available=[]
        for plane in range(4):
            part=scores[planes==plane]
            available.append(len(part)>0)
            pooled.append(part.topk(min(2,len(part)),dim=0).values.mean(0) if len(part) else torch.zeros_like(scores[0]))
        mask=torch.tensor(available,device=images.device)
        weights=self.plane_weight.masked_fill(~mask[:,None],-1e4).softmax(0)
        return (weights*torch.stack(pooled)).sum(0)

# Check both checkpoint compatibility and attention-grid geometry before touching DICOM data.
_probe_model=StudyCoAtNet(load_pretrained=True).to(DEVICE).eval()
with torch.inference_mode():
    _probe_x=torch.zeros((1,3,IMAGE_SIZE,IMAGE_SIZE),dtype=torch.uint8,device=DEVICE)
    _probe_plane=torch.zeros(1,dtype=torch.long,device=DEVICE)
    _probe_y=_probe_model(_probe_x,_probe_plane)
assert tuple(_probe_y.shape)==(len(LABEL_COLS),), tuple(_probe_y.shape)
del _probe_model,_probe_x,_probe_plane,_probe_y
if DEVICE.type=='cuda': torch.cuda.empty_cache()
print(f"CoAtNet checkpoint + {IMAGE_SIZE}x{IMAGE_SIZE} forward pass OK; final stage will fine-tune")


CoAtNet checkpoint + 224x224 forward pass OK; final stage will fine-tune


In [5]:
def plane_of(row):
    p=str(row.get('Anatomical_Plane','')).strip().upper()
    if p.startswith('SAG'): return 'SAGITTAL'
    if p.startswith('COR'): return 'CORONAL'
    if p.startswith('AX'): return 'AXIAL'
    return 'UNKNOWN'

def select_series(df, ids):
    df=df.copy()
    df['StudyInstanceUID']=df.StudyInstanceUID.astype(str)
    df['SeriesInstanceUID']=df.SeriesInstanceUID.astype(str)
    df=df[df.StudyInstanceUID.isin(set(map(str,ids)))].drop_duplicates(['StudyInstanceUID','SeriesInstanceUID'])
    df['plane_norm']=df.apply(plane_of,axis=1)
    for col in ('Fluid_Sensitive','Fat_Suppression'):
        if col not in df: df[col]='UNKNOWN'
        df[col]=df[col].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    rows=[]
    for sid,group in df.groupby('StudyInstanceUID',sort=True):
        group=group.sort_values('SeriesInstanceUID')
        picks=[]
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            if len(candidates): picks.append(candidates.iloc[0])
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            used=[p for p in picks if p.plane_norm==plane]
            if len(candidates)<2 or not used: continue
            first=used[0]
            rest=candidates[candidates.SeriesInstanceUID.ne(first.SeriesInstanceUID)]
            diverse=rest[(rest.Fluid_Sensitive.ne(first.Fluid_Sensitive)) | (rest.Fat_Suppression.ne(first.Fat_Suppression))]
            if len(diverse): picks.append(diverse.iloc[0])
            elif len(rest): picks.append(rest.iloc[0])
        if not picks and len(group): picks.append(group.iloc[0])
        rows.extend([{'StudyInstanceUID':sid,'SeriesInstanceUID':str(r.SeriesInstanceUID),'plane_norm':r.plane_norm} for r in picks])
    return pd.DataFrame(rows,columns=['StudyInstanceUID','SeriesInstanceUID','plane_norm'])

train_picks=select_series(train_series,pd.concat([gold.StudyInstanceUID,pseudo.StudyInstanceUID],ignore_index=True))
test_picks=select_series(test_series,sample_submission.StudyInstanceUID)
train_picks.to_csv(OUT/f'{RUN_TAG}_series_selection.csv',index=False)
assert set(gold.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Gold study has no metadata series'
assert set(pseudo.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Pseudo study has no metadata series'
print('gold series',len(train_picks),'test series',len(test_picks))


gold series 20145 test series 14


In [6]:
def _read_one(path):
    try:
        d=pydicom.dcmread(str(path),force=True)
        x=np.squeeze(d.pixel_array)
        if x.ndim!=2: return None
        x=x.astype(np.float32)*float(getattr(d,'RescaleSlope',1))+float(getattr(d,'RescaleIntercept',0))
        if str(getattr(d,'PhotometricInterpretation',''))=='MONOCHROME1': x=x.max()-x
        o=np.asarray(getattr(d,'ImageOrientationPatient',[]),float)
        pos=np.asarray(getattr(d,'ImagePositionPatient',[]),float)
        g=np.nan
        if len(o)==6 and len(pos)==3:
            n=np.cross(o[:3],o[3:]); norm=np.linalg.norm(n)
            if norm>0: g=float(pos @ (n/norm))
        try: inst=float(d.InstanceNumber)
        except Exception: inst=np.nan
        spacing=None
        try:
            ps=np.asarray(d.PixelSpacing,dtype=float)
            if ps.shape==(2,) and np.isfinite(ps).all() and (ps>0).all(): spacing=tuple(ps)
        except (AttributeError,ValueError,TypeError): pass
        return (path.name,g,inst,x,spacing)
    except Exception: return None

def center_crop_resize(x,spacing):
    # PixelSpacing is [row spacing, column spacing], in millimetres.
    h,w=x.shape
    if spacing is not None:
        ch,cw=max(1,round(CENTER_CROP_MM/spacing[0])),max(1,round(CENTER_CROP_MM/spacing[1]))
    else:
        ch=cw=max(1,round(min(h,w)*FALLBACK_CENTER_FRACTION))
    # Cap an implausibly large requested canvas; corrupt spacing uses the fallback.
    if ch>4*h or cw>4*w:
        ch=cw=max(1,round(min(h,w)*FALLBACK_CENTER_FRACTION))
    y0=(h-ch)//2; x0=(w-cw)//2
    out=np.zeros((ch,cw),np.float32)
    sy0,sy1=max(0,y0),min(h,y0+ch); sx0,sx1=max(0,x0),min(w,x0+cw)
    out[sy0-y0:sy1-y0,sx0-x0:sx1-x0]=x[sy0:sy1,sx0:sx1]
    z=cv2.resize(out,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_AREA if max(ch,cw)>IMAGE_SIZE else cv2.INTER_LINEAR)
    return np.rint(np.clip(z,0,1)*255).astype(np.uint8)

import hashlib, os, shutil, tempfile

# Cache cropped uint8 triplets on disk; augmentation is applied only after loading.
GOLD_IDS=set(gold.StudyInstanceUID.astype(str))
GOLD_SERIES_RAM={}
FAILED_SERIES_KEYS=set()
CACHE_ROOT.mkdir(parents=True,exist_ok=True)

def series_key(study,series,root):
    return (str(Path(root).resolve()),str(study),str(series))

def cache_path(key):
    contract=f'rsna-triplets-v2|size={IMAGE_SIZE}|centers={CENTERS_PER_PLANE}|cropmm={CENTER_CROP_MM}|fallback={FALLBACK_CENTER_FRACTION}|p1-p99-stride8|physical-sort|'+ '|'.join(key)
    digest=hashlib.sha256(contract.encode()).hexdigest()
    return CACHE_ROOT/f'{digest}.npy'

def decode_series(study,series,root):
    directory=root/str(study)/str(series)
    paths=sorted(p for p in directory.iterdir() if p.is_file() and not p.name.startswith('.')) if directory.is_dir() else []
    if not paths: return None
    with ThreadPoolExecutor(max_workers=NUM_DICOM_THREADS) as pool:
        rows=[r for r in pool.map(_read_one,paths) if r is not None]
    if not rows: return None
    key=1 if all(np.isfinite(r[1]) for r in rows) else 2 if all(np.isfinite(r[2]) for r in rows) else 0
    rows.sort(key=lambda r:(r[key],r[0]) if key else (r[0],))
    samples=np.concatenate([r[3][::8,::8].ravel() for r in rows])
    samples=samples[np.isfinite(samples)]
    if not len(samples): return None
    lo,hi=np.percentile(samples,[1,99])
    n=len(rows)
    centers=np.rint(np.linspace(.05*(n-1),.95*(n-1),CENTERS_PER_PLANE)).astype(int)
    indices=np.stack([np.maximum(0,centers-1),centers,np.minimum(n-1,centers+1)],axis=1)
    pixels={}
    for i in np.unique(indices):
        row=rows[i]
        x=np.nan_to_num(row[3],nan=lo,posinf=hi,neginf=lo)
        x=(np.clip(x,lo,hi)-lo)/(hi-lo) if hi>lo else np.zeros_like(x)
        pixels[int(i)]=center_crop_resize(x.astype(np.float32),row[4])
    # Fixed 10 positions, each with its two actual neighbouring slices as channels.
    stack=np.stack([np.stack([pixels[int(i)] for i in group]) for group in indices])
    stack.flags.writeable=False
    return stack

def get_series_stack(study,series,root):
    key=series_key(study,series,root)
    if key in GOLD_SERIES_RAM: return GOLD_SERIES_RAM[key]
    if key in FAILED_SERIES_KEYS: return None
    is_gold=key[0]==str(TRAIN_SERIES_ROOT.resolve()) and str(study) in GOLD_IDS
    if not is_gold:
        raise RuntimeError('Weak/test series must use the temporary DISK cache in load_plane.')
    path=cache_path(key)
    if path.is_file():
        try:
            stack=np.load(path,mmap_mode='r',allow_pickle=False)
            if stack.dtype!=np.uint8 or stack.shape!=(CENTERS_PER_PLANE,3,IMAGE_SIZE,IMAGE_SIZE):
                raise ValueError('Invalid cached gold stack')
            return stack
        except (ValueError,OSError):
            path.unlink(missing_ok=True)
    stack=decode_series(study,series,root)
    if stack is None: return None
    free=shutil.disk_usage(CACHE_ROOT).free
    if free<=stack.nbytes+CACHE_MIN_FREE_GIB*1024**3:
        raise OSError('Not enough disk space for the gold cache. Free space in /kaggle/working and rerun cell 6.')
    temp_path=None
    try:
        with tempfile.NamedTemporaryFile(dir=CACHE_ROOT,suffix='.tmp',delete=False) as f:
            temp_path=Path(f.name)
            np.save(f,stack,allow_pickle=False)
        os.replace(temp_path,path)
    finally:
        if temp_path is not None: temp_path.unlink(missing_ok=True)
    return stack

def sample_series(stack,training=False):
    # Epochs read preprocessed disk pixels, never DICOM files.
    return np.asarray(stack).copy()

WEAK_CACHE_ROOT=CACHE_ROOT/'active_weak_fold'
WEAK_CACHE_ROOT.mkdir(parents=True,exist_ok=True)
ACTIVE_WEAK_CACHE={}

def clear_weak_fold_cache():
    ACTIVE_WEAK_CACHE.clear()
    # Dedicated active-fold directory; never remove gold files or checkpoints.
    for path in WEAK_CACHE_ROOT.glob('*.npy'): path.unlink()
    for path in WEAK_CACHE_ROOT.glob('*.tmp'): path.unlink()

def prepare_weak_fold_cache(frame,picks=train_picks,source_root=TRAIN_SERIES_ROOT):
    clear_weak_fold_cache()
    ids=set(frame.StudyInstanceUID.astype(str))
    rows=picks[picks.StudyInstanceUID.isin(ids)].drop_duplicates(['StudyInstanceUID','SeriesInstanceUID'])
    tasks=[(str(r.StudyInstanceUID),str(r.SeriesInstanceUID)) for r in rows.itertuples(index=False)]
    started=time.time(); total_bytes=0; good_studies=set()
    print(f'Building disk cache for {len(ids)} studies / {len(tasks)} series before training...',flush=True)
    def build_one(task):
        sid,series=task; key=series_key(sid,series,source_root)
        stack=decode_series(sid,series,source_root)
        if stack is None: return key,None,0
        if shutil.disk_usage(WEAK_CACHE_ROOT).free<=stack.nbytes+CACHE_MIN_FREE_GIB*1024**3:
            raise OSError('This fold subset exceeds available disk space. Free disk space or reduce CENTERS_PER_PLANE and adjust caching before rerunning.')
        path=WEAK_CACHE_ROOT/cache_path(key).name
        tmp=None
        try:
            with tempfile.NamedTemporaryFile(dir=WEAK_CACHE_ROOT,suffix='.tmp',delete=False) as f:
                tmp=Path(f.name);np.save(f,stack,allow_pickle=False)
            os.replace(tmp,path)
        finally:
            if tmp is not None:tmp.unlink(missing_ok=True)
        size=stack.nbytes;del stack
        return key,path,size
    try:
        with ThreadPoolExecutor(max_workers=CACHE_BUILD_WORKERS) as pool:
            for count,(key,path,size) in enumerate(pool.map(build_one,tasks),1):
                ACTIVE_WEAK_CACHE[key]=path
                total_bytes+=size
                if path is not None:good_studies.add(key[1])
                if count%50==0 or count==len(tasks):
                    print(f'Fold cache {count}/{len(tasks)} series; {total_bytes/1024**3:.2f} GiB; {(time.time()-started)/60:.1f}m',flush=True)
        missing=ids-good_studies
        if missing:raise RuntimeError(f'Fold subset has {len(missing)} studies with no usable pixels: {sorted(missing)[:5]}')
    except BaseException:
        clear_weak_fold_cache()
        raise
    print('Fold subset fully cached. These files remain through all five epochs.',flush=True)

def load_plane(study,series,root,training=False):
    key=series_key(study,series,root)
    is_gold=key[0]==str(TRAIN_SERIES_ROOT.resolve()) and str(study) in GOLD_IDS
    if is_gold:
        stack=get_series_stack(study,series,root)
        if stack is None:return None
        return sample_series(stack,training=training)
    if key not in ACTIVE_WEAK_CACHE:
        raise RuntimeError(f'Study {study} has not been cached in the active fold. No DICOM fallback during training.')
    path=ACTIVE_WEAK_CACHE[key]
    if path is None:return None
    stack=np.load(path,mmap_mode='r',allow_pickle=False)
    try:return sample_series(stack,training=training)
    finally:stack._mmap.close()

def augment_study(images):
    # One sampled transform for the entire study, including neighbouring-slice channels.
    h,w=images.shape[-2:]
    angle=np.random.uniform(-AUG_ROTATION_DEG,AUG_ROTATION_DEG)
    matrix=cv2.getRotationMatrix2D(((w-1)/2,(h-1)/2),angle,1.)
    matrix[:,2]+=np.random.uniform(-AUG_TRANSLATION,AUG_TRANSLATION,2)*np.array([w,h])
    gain=np.random.uniform(*AUG_GAIN); offset=np.random.uniform(-AUG_OFFSET,AUG_OFFSET)
    output=np.empty_like(images)
    for i,image in enumerate(images):
        warped=cv2.warpAffine(image.transpose(1,2,0),matrix,(w,h),
                              flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT,borderValue=0)
        valid=cv2.warpAffine(np.ones((h,w),np.uint8),matrix,(w,h),
                            flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT,borderValue=0)
        transformed=np.clip(warped.astype(np.float32)*gain+offset*255,0,255)
        transformed[valid==0]=0
        output[i]=np.rint(transformed).astype(np.uint8).transpose(2,0,1)
    return output

def load_study(study,picks,root,training=False):
    """Gold from resident cache; weak/test from the fully prepared active fold."""
    blocks=[]; planes=[]
    group=picks[picks.StudyInstanceUID.eq(str(study))]
    for row in group.itertuples(index=False):
        block=load_plane(study,row.SeriesInstanceUID,root,training=training)
        if block is None: continue
        blocks.append(block)
        plane_id=('SAGITTAL','CORONAL','AXIAL').index(row.plane_norm) if row.plane_norm in ('SAGITTAL','CORONAL','AXIAL') else 3
        planes.extend([plane_id]*len(block))
    if not blocks: return None
    images=np.concatenate(blocks,0)
    if training: images=augment_study(images)
    return images,np.asarray(planes,np.int64)

# Keep only gold files in the notebook's pixel-cache directory.
# This frees files written by the previous eager-all-studies version.
gold_cache_names={cache_path(series_key(row.StudyInstanceUID,row.SeriesInstanceUID,TRAIN_SERIES_ROOT)).name
                  for row in train_picks[train_picks.StudyInstanceUID.isin(GOLD_IDS)].itertuples(index=False)}
if CLEAR_OLD_WEAK_CACHE:
    import re
    removed=0; freed=0
    for path in CACHE_ROOT.glob('*.npy'):
        if re.fullmatch(r'[0-9a-f]{64}\.npy',path.name) and path.name not in gold_cache_names:
            freed+=path.stat().st_size; path.unlink(); removed+=1
    print(f'Removed {removed} old non-gold pixel cache files; freed {freed/1024**3:.2f} GiB')

# All 58 gold studies are loaded once before worker processes and fold training.
print('Loading 58 gold studies into RAM once...')
clear_weak_fold_cache()
# Remove obsolete per-series temporary files from the prior implementation.
old_group=CACHE_ROOT/'active_weak_chunk'
if old_group.is_dir():
    for abandoned in old_group.glob('*.npy'):abandoned.unlink()
    for abandoned in old_group.glob('*.tmp'):abandoned.unlink()
old_temporary=CACHE_ROOT/'active_weak_series'
if old_temporary.is_dir():
    for abandoned in old_temporary.glob('tmp*.npy'):abandoned.unlink()
for count,sid in enumerate(gold.StudyInstanceUID.astype(str),1):
    loaded=0
    for row in train_picks[train_picks.StudyInstanceUID.eq(sid)].itertuples(index=False):
        key=series_key(sid,row.SeriesInstanceUID,TRAIN_SERIES_ROOT)
        stack=get_series_stack(sid,row.SeriesInstanceUID,TRAIN_SERIES_ROOT)
        if stack is None:
            FAILED_SERIES_KEYS.add(key)
            continue
        # A normal immutable ndarray keeps gold resident, including when cache was mmap-backed.
        resident=np.array(stack,copy=True); resident.flags.writeable=False
        GOLD_SERIES_RAM[key]=resident
        del stack,resident
        loaded+=1
    if loaded==0: raise RuntimeError(f'Could not decode gold study {sid}')
    if count%10==0 or count==len(gold):
        gib=sum(x.nbytes for x in GOLD_SERIES_RAM.values())/1024**3
        print(f'Gold loaded {count}/{len(gold)}; resident series pixels {gib:.2f} GiB')
print('Gold-only disk cache:',CACHE_ROOT)
print('Weak studies split across models: 800, 800, 800, 800, 800. Cache each subset once for all 5 epochs, then delete it.')
print('Gold stays resident across all models. Only one weak-fold subset occupies disk at a time.')


Removed 0 old non-gold pixel cache files; freed 0.00 GiB
Loading 58 gold studies into RAM once...
Gold loaded 10/58; resident series pixels 0.07 GiB
Gold loaded 20/58; resident series pixels 0.14 GiB
Gold loaded 30/58; resident series pixels 0.21 GiB
Gold loaded 40/58; resident series pixels 0.28 GiB
Gold loaded 50/58; resident series pixels 0.35 GiB
Gold loaded 58/58; resident series pixels 0.41 GiB
Gold-only disk cache: /kaggle/working/rsna_coatnet_crop140mm_10triplet_cache_v2
Weak studies split across models: 800, 800, 800, 800, 800. Cache each subset once for all 5 epochs, then delete it.
Gold stays resident across all models. Only one weak-fold subset occupies disk at a time.


In [7]:

def first_item(batch):
    # Top-level function is safe with multiprocessing spawn/fork; lambdas are not.
    return batch[0]

def save_compact_checkpoint(model,path):
    # Full state makes test inference independent of the initial ImageNet file.
    torch.save({'model_state':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                'model_name':MODEL_NAME,'run_tag':RUN_TAG},path)

def load_compact_checkpoint(model,path):
    state=torch.load(path,map_location='cpu',weights_only=True)
    if state.get('model_name')!=MODEL_NAME or state.get('run_tag')!=RUN_TAG:
        raise ValueError('Checkpoint belongs to a different experiment')
    model.load_state_dict(state['model_state'],strict=True)
    return model

class BagDS(Dataset):
    def __init__(self,frame,training=True): self.frame=frame.reset_index(drop=True); self.training=training
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        loaded=load_study(str(row.StudyInstanceUID),train_picks,TRAIN_SERIES_ROOT,training=self.training)
        if loaded is None:
            raise RuntimeError(f'Could not decode study {row.StudyInstanceUID}')
        x,plane=loaded
        return torch.from_numpy(x),torch.from_numpy(plane),torch.tensor(row[LABEL_COLS].to_numpy(np.float32))

class PseudoBagDS(BagDS):
    def __getitem__(self,i):
        x,plane,y=super().__getitem__(i)
        row=self.frame.iloc[i]
        weight=torch.tensor(row[['weight__'+c for c in LABEL_COLS]].to_numpy(np.float32))
        return x,plane,y,weight

def weak_epoch_indices(n,epochs,passes,seed):
    if min(n,epochs,passes)<1: raise ValueError('Counts must be positive')
    rng=np.random.default_rng(seed)
    stream=np.concatenate([rng.permutation(n) for _ in range(passes)])
    chunks=np.array_split(stream,epochs)
    if any(len(chunk)==0 for chunk in chunks): raise ValueError('Too many epochs for the weak-data presentations')
    assert np.all(np.bincount(stream,minlength=n)==passes)
    return chunks

def train_fold(model,frame,validation,fold,checkpoint,fold_pseudo):
    gold_frame=pd.concat([frame]*GOLD_REPEATS,ignore_index=True)
    gold_loader=DataLoader(BagDS(gold_frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                      **({'prefetch_factor':1} if NUM_WORKERS>0 else {}),
                      collate_fn=first_item)
    print("trainable backbone parameters",sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
    optimizer=torch.optim.AdamW([{'params':[p for p in model.backbone.parameters() if p.requires_grad],'lr':LR_BACKBONE},
                                 {'params':list(model.head.parameters())+[model.plane_weight],'lr':LR_HEAD}],weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
    best_bce=np.inf; best_epoch=None; history=[]
    # One full pass per epoch through this model's disjoint weak subset; gold validation IDs remain held out.
    weak_chunks=weak_epoch_indices(len(fold_pseudo),EPOCHS,WEAK_PASSES,SEED)
    total_steps=sum(len(chunk)+len(gold_frame) for chunk in weak_chunks)
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,
        lr_lambda=lambda step: MIN_LR_RATIO+(1-MIN_LR_RATIO)*.5*(1+np.cos(np.pi*min(step,total_steps)/total_steps)))
    print('Weak presentations:',len(fold_pseudo)*WEAK_PASSES,'gold presentations:',len(gold_frame)*EPOCHS)
    print('Validation/test: fixed slices and no augmentation.')
    for epoch in range(1,EPOCHS+1):
        model.train(); model.train_backbone_tail()
        losses=[]; start=time.time()
        epoch_lrs=[group['lr'] for group in optimizer.param_groups]
        weak_frame=fold_pseudo.iloc[weak_chunks[epoch-1]]
        assert len(weak_frame)>0
        weak_loader=DataLoader(PseudoBagDS(weak_frame),batch_size=1,shuffle=True,num_workers=0,
                          pin_memory=DEVICE.type=='cuda',persistent_workers=False,
                          collate_fn=first_item)
        gold_iter=iter(gold_loader); weak_iter=iter(weak_loader)
        schedule=np.array([0]*len(gold_frame)+[1]*len(weak_frame))
        np.random.default_rng(SEED+fold*100+epoch).shuffle(schedule)
        for mode in schedule:
            if mode==0:
                x,plane,y=next(gold_iter); weights=None
            else:
                x,plane,y,weights=next(weak_iter)
            x=x.to(DEVICE,non_blocking=True); plane=plane.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,plane).float()
            if mode==0:
                loss=nn.functional.binary_cross_entropy_with_logits(logits,y)
            else:
                weights=weights.to(DEVICE)
                raw=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
                # Apply source-specific confidence weights; keep the original 12-target denominator.
                loss=PSEUDO_LOSS_WEIGHT*(raw*weights).sum()/weights.numel()
            if not torch.isfinite(loss): raise RuntimeError(f'Nonfinite loss in fold {fold}, epoch {epoch}')
            scaler.scale(loss).backward()
            previous_scale=scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale()>=previous_scale: scheduler.step()
            losses.append(loss.item())
        pred,failed=predict_ids(model,validation.StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
        if failed: raise RuntimeError(f'Fold {fold} validation decode failures: {failed}')
        aligned=validation[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
        y=validation[LABEL_COLS].to_numpy(np.float32)
        probs=aligned[LABEL_COLS].to_numpy(np.float32).clip(1e-6,1-1e-6)
        val_bce=nn.functional.binary_cross_entropy(torch.from_numpy(probs),torch.from_numpy(y)).item()
        aucs=[roc_auc_score(y[:,j],probs[:,j]) for j in range(len(LABEL_COLS)) if np.unique(y[:,j]).size==2]
        val_auc=float(np.mean(aucs)) if aucs else float('nan')
        if val_bce<best_bce-1e-6:
            best_bce=val_bce; best_epoch=epoch
            save_compact_checkpoint(model,checkpoint)
        history.append({'fold':fold,'epoch':epoch,'train_weighted_loss':float(np.mean(losses)),
                        'val_bce':val_bce,'val_macro_auc':val_auc,'n_auc_labels':len(aucs),
                        'best_so_far':best_epoch,'weak_presentations':len(weak_frame),'gold_presentations':len(gold_frame),
                        'backbone_lr':epoch_lrs[0],'head_lr':epoch_lrs[1],'minutes':(time.time()-start)/60})
        print(f'fold {fold} epoch {epoch}/{EPOCHS}: weighted loss {np.mean(losses):.4f}, '
              f'val BCE {val_bce:.4f}, val AUC {val_auc:.4f} ({len(aucs)} labels), '
              f'best epoch {best_epoch}, lr {epoch_lrs[0]:.2g}/{epoch_lrs[1]:.2g}, time {(time.time()-start)/60:.1f}m')
    load_compact_checkpoint(model,checkpoint)
    return best_epoch,best_bce,history

@torch.inference_mode()
def predict_ids(model,ids,picks,root):
    model.eval(); rows=[]; failed=[]
    for sid in map(str,ids):
        loaded=load_study(sid,picks,root)
        if loaded is None:
            failed.append(sid); continue
        images,planes=loaded
        x=torch.from_numpy(images).to(DEVICE); plane=torch.from_numpy(planes).to(DEVICE)
        score=torch.sigmoid(model(x,plane).float()).cpu().numpy()
        rows.append([sid,*score.tolist()])
    return pd.DataFrame(rows,columns=['StudyInstanceUID']+LABEL_COLS),failed

folds=KFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
oof=np.full((len(gold),len(LABEL_COLS)),np.nan,np.float32)
fold_scores=[]; epoch_history=[]
# Disjoint weak partitions: every selected weak study belongs to exactly one model.
weak_fold_indices=np.array_split(np.random.default_rng(SEED).permutation(len(pseudo)),N_FOLDS)
assert sorted(np.concatenate(weak_fold_indices).tolist())==list(range(len(pseudo)))
print('Weak subset sizes:',[len(part) for part in weak_fold_indices])
weak_manifest=[]
for fold,part in enumerate(weak_fold_indices,1):
    weak_manifest.extend({'fold':fold,'StudyInstanceUID':str(sid)} for sid in pseudo.iloc[part].StudyInstanceUID)
pd.DataFrame(weak_manifest).to_csv(OUT/f'{RUN_TAG}_weak_fold_assignment.csv',index=False)
for fold,(tr,va) in enumerate(folds.split(gold),1):
    random.seed(SEED+fold); np.random.seed(SEED+fold); torch.manual_seed(SEED+fold)
    print('\nFOLD',fold,'train',len(tr),'validation',len(va))
    fold_pseudo=pseudo.iloc[weak_fold_indices[fold-1]].reset_index(drop=True)
    print('Weak studies for this model:',len(fold_pseudo),'| epochs:',EPOCHS)
    prepare_weak_fold_cache(fold_pseudo)
    try:
        model=StudyCoAtNet(load_pretrained=True).to(DEVICE)
        checkpoint=OUT/f'{RUN_TAG}_fold{fold}.pt'
        best_epoch,best_bce,history=train_fold(model,gold.iloc[tr],gold.iloc[va],fold,checkpoint,fold_pseudo)
        epoch_history.extend(history)
        pred,failed=predict_ids(model,gold.iloc[va].StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
        assert not failed
        aligned=gold.iloc[va][['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
        oof[va]=aligned[LABEL_COLS].to_numpy(np.float32)
        fold_scores.append({'fold':fold,'n_weak':len(fold_pseudo),'n_train':len(tr),'n_validation':len(va),'checkpoint':str(checkpoint),'best_epoch':best_epoch,'best_val_bce':best_bce})
        print('saved',checkpoint)
        del model; gc.collect()
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    finally:
        clear_weak_fold_cache()
        print(f'Fold {fold} finished: weak subset disk cache deleted.',flush=True)
assert np.isfinite(oof).all()
oof_df=gold[['StudyInstanceUID']+LABEL_COLS].copy()
for j,label in enumerate(LABEL_COLS): oof_df['pred__'+label]=oof[:,j]
oof_df.to_csv(OUT/f'{RUN_TAG}_oof_58.csv',index=False)
pd.DataFrame(fold_scores).to_csv(OUT/f'{RUN_TAG}_fold_manifest.csv',index=False)
pd.DataFrame(epoch_history).to_csv(OUT/f'{RUN_TAG}_epoch_history.csv',index=False)
metrics=[]
for j,label in enumerate(LABEL_COLS):
    y=gold[label].to_numpy(float)
    metrics.append({'label':label,'positives':int(y.sum()),'auc':roc_auc_score(y,oof[:,j]) if np.unique(y).size==2 else np.nan})
metrics=pd.DataFrame(metrics); metrics.to_csv(OUT/f'{RUN_TAG}_oof_auc.csv',index=False)
display(metrics); print(f'{N_FOLDS}-fold OOF macro AUC:',metrics.auc.mean())


Weak subset sizes: [800, 800, 800, 800, 800]

FOLD 1 train 46 validation 12
Weak studies for this model: 800 | epochs: 5
Building disk cache for 800 studies / 3980 series before training...
Fold cache 50/3980 series; 0.07 GiB; 0.3m
Fold cache 100/3980 series; 0.14 GiB; 0.6m
Fold cache 150/3980 series; 0.21 GiB; 0.9m
Fold cache 200/3980 series; 0.28 GiB; 1.2m
Fold cache 250/3980 series; 0.35 GiB; 1.4m
Fold cache 300/3980 series; 0.42 GiB; 1.7m
Fold cache 350/3980 series; 0.49 GiB; 2.0m
Fold cache 400/3980 series; 0.56 GiB; 2.2m
Fold cache 450/3980 series; 0.63 GiB; 2.5m
Fold cache 500/3980 series; 0.70 GiB; 2.8m
Fold cache 550/3980 series; 0.77 GiB; 3.0m
Fold cache 600/3980 series; 0.84 GiB; 3.5m
Fold cache 650/3980 series; 0.91 GiB; 3.8m
Fold cache 700/3980 series; 0.98 GiB; 4.1m
Fold cache 750/3980 series; 1.05 GiB; 4.4m
Fold cache 800/3980 series; 1.12 GiB; 4.6m
Fold cache 850/3980 series; 1.19 GiB; 4.8m
Fold cache 900/3980 series; 1.26 GiB; 5.2m
Fold cache 950/3980 series; 1.33 GiB;

,label,positives,auc
0,ACL,24,0.640931
1,MCL,9,0.514739
2,Medial Meniscus,26,0.691106
3,Lateral Meniscus,23,0.739130
4,Medial OA,15,0.844961
5,Lateral OA,11,0.719536
6,PF OA,21,0.740026
7,Effusion,35,0.877019
8,Synovitis,27,0.624851
9,Baker's,12,0.737319


5-fold OOF macro AUC: 0.7023589611729822


In [8]:
# Test data also uses complete <=400-study disk chunks, deleted after predicting each chunk.
test_ids=sample_submission.StudyInstanceUID.astype(str).tolist()
ensemble=np.zeros((len(test_ids),len(LABEL_COLS)),np.float64)
for fold in range(1,N_FOLDS+1):
    model=StudyCoAtNet(load_pretrained=False)
    load_compact_checkpoint(model,OUT/f'{RUN_TAG}_fold{fold}.pt')
    model=model.to(DEVICE)
    if DEVICE.type=='cuda': model=model.half()  # half weights and half inputs, no autocast at inference
    test_parts=[];failed=[]
    for start in range(0,len(test_ids),TEST_CACHE_GROUP_SIZE):
        group_ids=test_ids[start:start+TEST_CACHE_GROUP_SIZE]
        prepare_weak_fold_cache(pd.DataFrame({'StudyInstanceUID':group_ids}),test_picks,TEST_SERIES_ROOT)
        try:
            part,missing=predict_ids(model,group_ids,test_picks,TEST_SERIES_ROOT)
            test_parts.append(part);failed.extend(missing)
        finally:clear_weak_fold_cache()
    pred=pd.concat(test_parts,ignore_index=True) if test_parts else pd.DataFrame(columns=['StudyInstanceUID']+LABEL_COLS)
    aligned=sample_submission[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',how='left',validate='one_to_one')
    fallback=gold[LABEL_COLS].mean().to_numpy(float)
    for j,label in enumerate(LABEL_COLS): aligned[label]=aligned[label].fillna(float(fallback[j]))
    ensemble+=aligned[LABEL_COLS].to_numpy(float)/N_FOLDS
    print('fold',fold,'predicted',len(pred),'missing',len(failed))
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
submission=sample_submission[['StudyInstanceUID']].copy()
submission[LABEL_COLS]=ensemble.astype(np.float32)
assert list(submission.columns)==list(sample_submission.columns)
assert submission.StudyInstanceUID.is_unique and len(submission)==len(sample_submission)
assert np.isfinite(ensemble).all() and ensemble.min()>=0 and ensemble.max()<=1
submission.to_csv(OUT/'submission.csv',index=False)
print('wrote',OUT/'submission.csv','shape',submission.shape)
display(submission.head())


Building disk cache for 3 studies / 14 series before training...
Fold cache 14/14 series; 0.02 GiB; 0.1m
Fold subset fully cached. These files remain through all five epochs.
fold 1 predicted 3 missing 0
Building disk cache for 3 studies / 14 series before training...
Fold cache 14/14 series; 0.02 GiB; 0.1m
Fold subset fully cached. These files remain through all five epochs.
fold 2 predicted 3 missing 0
Building disk cache for 3 studies / 14 series before training...
Fold cache 14/14 series; 0.02 GiB; 0.1m
Fold subset fully cached. These files remain through all five epochs.
fold 3 predicted 3 missing 0
Building disk cache for 3 studies / 14 series before training...
Fold cache 14/14 series; 0.02 GiB; 0.1m
Fold subset fully cached. These files remain through all five epochs.
fold 4 predicted 3 missing 0
Building disk cache for 3 studies / 14 series before training...
Fold cache 14/14 series; 0.02 GiB; 0.1m
Fold subset fully cached. These files remain through all five epochs.
fold 5 pr

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.150214,0.120306,0.387141,0.201241,0.456380,0.367925,0.606033,0.526231,0.529936,0.416033,0.242414,0.071510
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.118871,0.159955,0.247980,0.182277,0.338085,0.253398,0.571342,0.289147,0.419236,0.201570,0.209047,0.089120
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.112895,0.219267,0.524897,0.305792,0.559186,0.419434,0.707858,0.391531,0.486918,0.389610,0.150530,0.155192
